# Block 06: Scaled dot-product attention and the causal mask

**Paper anchor:** Eq. 1, §3.2.1, §3.2.3 (third bullet: masking), Figure 2 (left), footnote 4
**Builds on:** block 03 (softmax); prerequisite P5
**Time:** about 6 hours, in short sessions
**You will write:** `causal_mask`, `scaled_dot_product_attention`, the footnote 4 experiment (stretch: backward by hand)
**Done when:** both checkpoints pass and you can draw Figure 2 (left) from memory with shapes

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b06
from minitransformer.numpy_ops import softmax  # graduated in block 03

## Big picture
Attention is a learned weighted average. Each query scores every key, softmax turns the scores into weights,
and the output is the weighted sum of the values (§3.2):

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V \qquad \text{(Eq. 1)}$$

| tensor | shape |
|---|---|
| Q | (..., T_q, d_k) |
| K | (..., T_k, d_k) |
| V | (..., T_k, d_v) |
| scores $QK^\top/\sqrt{d_k}$ | (..., T_q, T_k) |
| out | (..., T_q, d_v) |

**Notation traps.** Rows are positions (row-vector convention). In this repo a boolean mask uses **True = allowed**.
The mask acts on the scores before the softmax: illegal entries become $-\infty$ (§3.2.3, the "Mask (opt.)" box in Fig. 2).

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. What shape is $QK^\top$ for T = 5, d_k = 8? What does entry [i, j] mean?
2. Why must the mask be applied before the softmax rather than after?
3. Footnote 4: if q and k have independent N(0, 1) entries, what are the mean and variance of $q \cdot k$?

*My answers:*

1. 
2. 
3. 

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def causal_mask(T):
    """Boolean (T, T) mask with M[i, j] = True exactly when j <= i.

    Row = query position, column = key position. True = allowed.
    """
    raise NotImplementedError("Block 06: your turn")

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Eq. 1: softmax(Q K^T / sqrt(d_k)) V, with optional masking (§3.2.3).

    Q:    (..., T_q, d_k)
    K:    (..., T_k, d_k)
    V:    (..., T_k, d_v)
    mask: None, or bool array broadcastable to (..., T_q, T_k); True = allowed.
          Disallowed scores become -inf (or a very large negative number) BEFORE the softmax.
    returns: (out, weights)
        out:     (..., T_q, d_v)
        weights: (..., T_q, T_k), each row sums to 1
    Leading axes (batch, heads) are optional and must broadcast.
    """
    raise NotImplementedError("Block 06: your turn")

In [ ]:
b06.check_causal_mask(causal_mask)
b06.check_attention(scaled_dot_product_attention)
b06.check_masked_attention(scaled_dot_product_attention, causal_mask)

## Toy example
T = 3, d_k = 2. Pick small integer Q, K, V, print scores, weights and output with and without the causal mask, and verify one row by hand.

In [ ]:
# YOUR CODE

## Experiment: why divide by $\sqrt{d_k}$? (footnote 4)
The paper suspects large $d_k$ pushes the softmax into regions with tiny gradients (§3.2.1) but never ablates it.
**Hypothesis:** for random unit-variance q, k, the variance of $q \cdot k$ is about $d_k$; without scaling, the mean
largest attention weight over T = 16 keys approaches 1 as $d_k$ grows, while with scaling it stays roughly constant.
**Falsified if** the unscaled largest weight does not increase with $d_k$ in 4, 16, 64, 256, 1024.

In [ ]:
# YOUR CODE

## Stretch: attention backward by hand
Only after the rest is solid. 2-D inputs only.

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def sdpa_forward_with_cache(Q, K, V, mask=None):
    """Same maths as scaled_dot_product_attention, 2-D inputs only, returning a cache.

    Q: (T_q, d_k), K: (T_k, d_k), V: (T_k, d_v), mask: None or bool (T_q, T_k), True = allowed
    returns (out (T_q, d_v), cache)
    """
    raise NotImplementedError("Block 06 stretch: your turn")

def sdpa_backward(dout, cache):
    """Given dL/dout (T_q, d_v), return (dQ, dK, dV) with the shapes of Q, K, V."""
    raise NotImplementedError("Block 06 stretch: your turn")

In [ ]:
b06.check_attention_backward(sdpa_forward_with_cache, sdpa_backward, causal_mask)

## Explain it back
Without notes. Then paste your answers into chat.

1. Explain Eq. 1 to someone who knows weighted averages but not attention.
2. Why the square root of d_k, and what breaks without it?
3. Where exactly does the causal mask act, and what would leak without it?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
Copy `causal_mask` and `scaled_dot_product_attention` into `numpy_ops.py`, run `pytest -q tests/test_numpy_ops.py`, commit `block 06: scaled dot-product attention`. Mark P5 done.